# TP4 : Implémentation d'un système RAG (Retrieval-Augmented Generation)

## Présentation du TP

Ce travaux pratiques (TP) est dédié à la mise en place et à l'évaluation d'une architecture **RAG** (*Retrieval-Augmented Generation*). L'objectif principal est de pallier les limites de connaissances d'un grand modèle de langage (LLM) en lui permettant d'extraire et de s'appuyer sur des informations issues de documents externes spécifiques avant de générer une réponse.

Au cours de ce TP, nous allons aborder les étapes clés suivantes :
* **Le Document Loading & Splitting** : Chargement et découpage des documents en blocs (*chunks*).
* **L'Indexation & Vectorisation** : Transformation des textes en vecteurs et stockage dans une base de données vectorielle.
* **Le Retrieval (Recherche)** : Extraction des segments de texte les plus pertinents vis-à-vis d'une requête utilisateur.
* **La Génération & Évaluation** : Injection du contexte dans le prompt du LLM et analyse de la fidélité de ses réponses.


# 0. Preparation de l'environement 

Dans le terminal `uv init` , ` uv venv `, ` Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process ` et ` .venv\Scripts\activate ` .  ` uv add ipykernel ` et  ` uv add langchain ` puis ` uv add langchain-openai ` et  `uv add langchain-text-splitters ` puis  `uv add langchain-community ` finalement pour importer la clee api open ai ` uv add python-dotenv  `


In [1]:
import json
import tiktoken
#import pandas as pd||
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader 
from langchain_community.vectorstores import Chroma
from dotenv.ipython import load_dotenv
import os

C:\Users\user\AppData\Local\Temp\ipykernel_7112\4814559.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader


##  LLM

In [2]:
load_dotenv(override=True)

True

In [3]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

# 1. Loading the pdf, Chunking

Je vais créer un dossier que je vais appeler `PDFs` et à l'intérieur duquel je vais coller ce fichier PDF `nc346.pdf` (pour vous, choisissez votre CV par exemple, ou un fichier que vous connaissez déjà).


In [4]:
pdf_file="./pdfs/cv.pdf"

In [5]:
pdf_loader=PyPDFLoader(pdf_file)

Et si vous voulez charger un dossier qui contient plusieurs fichiers, vous utilisez `PyPDFDirectoryLoader(path="./pdfs")`.

On va utiliser le découpage (*split*). Je rappelle donc que l'on utilise `RecursiveCharacterTextSplitter` avec l'encodeur `tiktoken`. Voici le modèle Tiktoken utilisé : la taille de chaque bloc (*chunk*) est de 300 tokens et le chevauchement (*overlap*) est de 20.

Maintenant, je vais charger le modèle `TextSplitter`. Notez qu'ici, il manque une dépendance. Pour effectuer le découpage, le système vous demande d'installer `pypdf`.

Faites donc : `uv add pypdf`.


In [6]:
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(encoding_name='o200k_base',chunk_size=300,chunk_overlap=20)

Dans le terminal de projet ajouter ` uv add pypdf `

La méthode `load_and_split` va charger le fichier, puis appliquer cette stratégie de découpage. Chaque bloc (*chunk*) aura une taille de 300 tokens avec un chevauchement (*overlap*) de 20. Si vous souhaitez vérifier le nombre total de blocs générés, vous pouvez utiliser `len(chunks)`.


In [7]:
chunks=pdf_loader.load_and_split(text_splitter)
len(chunks)


3

# 2.vectore Store - ChromaDB et Embedding

 *Importer le model d'embedding* 



In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

d:\AIT BOUZIAN IBTISSAM\MASTER  AIDC\S3\Partie1\Application de system destriber\TP4RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1493.04it/s]


In [9]:
vectorstore = Chroma.from_documents(
    chunks, embedding_model,
    collection_name="cvBTISSAM11",
    persist_directory="./store"
)

Ajouter un dossier "store" et utiliser la commande ` uv add chromadb ` pour ajouter *chromadb*

 Je vais vectoriser les chunks et je les stock dans la base de données vectorielle ici **vectorstore**

# 3.  **Trouver les chunks liée à la questions (Retriever)**

In [10]:
#crée le Retriever 
retriever=vectorstore.as_retriever(search_type='similarity',search_Kwargs={'k',5})
# k=5. Donc je vais lui demander de me chercher les cinq chunks les plus proches à la question.


On va faire un test. J'ai la question suivante : 
> « Quelle est la capitalisation boursière du Maroc pour 2025 ? »

La méthode `retriever.invoke()` a recherché les blocs (*chunks*). Je vais maintenant les afficher.


In [11]:
#test
retrieved_chunks=retriever.invoke(" QU'il est duree de formation ?")
print(retrieved_chunks)
print("nombre de chunks est :",len(retrieved_chunks))


[Document(metadata={'page': 0, 'moddate': '2026-10-15T16:21:17+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'page_label': '1', 'trapped': '/False', 'author': 'Ibtissam AIT BOUZIAN', 'source': './pdfs/cv.pdf', 'subject': '', 'creator': 'LaTeX with hyperref', 'keywords': '', 'total_pages': 1, 'title': 'CV — Ibtissam AIT BOUZIAN', 'creationdate': '2026-10-15T16:21:17+00:00', 'producer': 'pdfTeX-1.40.27'}, page_content='IBTISSAM AIT BOUZIAN\nÉtudiante en Master Intelligence Artificielle & Digital Computing (AIDC)\nRecherche un stage PFE de 4 mois en Intelligence Artificielle\n♂phone+212 607846288 |✉i.aitbouzian@gmail.com |♂¶ap-¶arker-altMaroc\nProfil\nÉtudiante en Master IA, passionnée par le machine learning, le deep learning et la vision par ordinateur. Expérience\nde solutions d’IA de bout en bout : entraînement de modèles, traitement de flux temps réel, déploiement d’API et\nintégration IoT.\nFormation\nMaster Inte

Maintenant, pour la partie RAG, on va créer notre prompt système (*system prompt*).


# 4.  *Prompt Design*

## 4.1 Prompt template

In [12]:
prompt_template="""
Answer the following question based only on provided context The context is about NOTE DE
CONJONCTURE DIRECTION DES ETUDES ET DES PREVISIONS FINANCIERES Décembre 2025
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
{context}
</context>
<question>
{question}
</question>
"""

### Note sur le contexte
Vous pouvez ajouter cette précision si vous savez que votre document concerne la Direction des études et des prévisions financières (DEPF). Vous indiquez alors le contexte suivant : *« Le contexte concerne la Note de conjoncture de la Direction des études et des prévisions financières de décembre 2025. »* C'est une phrase utile. Si vous préférez que le prompt reste général, quel que soit le fichier PDF, vous pouvez simplement retirer cette partie.

Le contexte est délimité par la balise `<context>`, et la question par la balise `<question>`.

### Gestion des hallucinations
Pour éviter les hallucinations, spécifiez dans le prompt : *« Si la réponse ne figure pas dans ce contexte, répondez par : "Je ne sais pas !" »*. Cela permet de régler efficacement le problème des hallucinations.


## 4.2 *Retrieving the Relevant Documents*

### Passage à la partie *retrieving*

Voici ma démarche : je récupère les documents, puis je les **convertis**. Pour chaque document (`relevant_document_chunks`), je sélectionne uniquement le contenu de la page (`d.page_content`), ce qui me génère une liste de textes. Enfin, nous allons concaténer cette liste de *chunks* de texte en utilisant le séparateur `". "`.


In [35]:
user_input="qu'elle est son age ? "

relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=" . ".join(context_list)


context_for_query

'IBTISSAM AIT BOUZIAN\n+212 607 846 288 | i.aitbouzian@gmail.com | Midelt, Maroc\nPROFIL\nEtudiante en Master Intelligence Artificielle & Digital Computing, passionnee par le machine learning, la vision par\nordinateur et le traitement de donnees en flux. A la recherche d’un stage en IA ou Data Engineering pour mettre en\npratique mes competences en developpement de systemes intelligents. date-naissance :07/11/2003\nFORMATION\nMaster Intelligence Artificielle & Digital Computing (AIDC)2025 – Present\nFST Beni Mellal\nLicence Informatique 2024 – 2025\nFST Beni Mellal\nDEUST MIPC (Mathematiques, Informatique, Physique, Chimie)2022 – 2024\nFST Beni Mellal\nEXPERIENCE PROFESSIONNELLE\nStage en Developpement Informatique – ATNERAvril – Juin 2025\n•Developpement d’une application web de gestion et de suivi du materiel et du personnel affectes aux chantiers\n(affectation, pointage, carburant, maintenance).\n•Conception de la base de donnees et developpement des interfaces utilisateur.\n•Techn

In [36]:
len(relevant_document_chunks)

4

## 4.3 Le prompt

Maintenant, on va injecter dans le prompt le contexte et la question en utilisant template format. Vous voyez donc pour template On a injecté le contexte et la question. Et puis donc le si vous voulez afficher le prompt. Qu'est-ce qu'il contient ? Il contient:

 1. le système message:  ( Answer the following question ............  Je ne sais pas! `` <context> ``).
 2. le contexte:  entre ` <context> ` et `` </context> `` .
 3. question:  entre ` <question> ` et ` </question> ` .




In [37]:
prompt=prompt_template.format(context=context_for_query,question=user_input)
print(prompt)


Answer the following question based only on provided context The context is about NOTE DE
CONJONCTURE DIRECTION DES ETUDES ET DES PREVISIONS FINANCIERES Décembre 2025
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
IBTISSAM AIT BOUZIAN
+212 607 846 288 | i.aitbouzian@gmail.com | Midelt, Maroc
PROFIL
Etudiante en Master Intelligence Artificielle & Digital Computing, passionnee par le machine learning, la vision par
ordinateur et le traitement de donnees en flux. A la recherche d’un stage en IA ou Data Engineering pour mettre en
pratique mes competences en developpement de systemes intelligents. date-naissance :07/11/2003
FORMATION
Master Intelligence Artificielle & Digital Computing (AIDC)2025 – Present
FST Beni Mellal
Licence Informatique 2024 – 2025
FST Beni Mellal
DEUST MIPC (Mathematiques, Informatique, Physique, Chimie)2022 – 2024
FST Beni Mellal
EXPERIENCE PROFE

Donc, ça c'est le prompt.

# 5. *Récupérer la réponse de la part de llm*

 Maintenant on va utiliser le prompt. On va poser cette question à LLM.

In [38]:
import sys
!uv pip install --python "{sys.executable}" langchain-groq

Checked 1 package in 1.94s


In [39]:
import os, requests
r = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"}
)
for m in sorted(x["id"] for x in r.json()["data"]):
    print(m)

allam-2-7b
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
qwen/qwen3.8-27b
whisper-large-v3
whisper-large-v3-turbo


In [40]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
groundness_checker = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

In [41]:
resp=llm.invoke(prompt)
from IPython.display import Markdown
print(display(Markdown(resp.content)))

Elle a 22 ans.

None


## 6. Regroupement dans une application RAG


### Définition de la fonction RAG

Nous allons définir `RAG` comme notre fonction principale de RAG. Elle prendra en paramètres : `query`, `llm` et `prompt_template`.

**Comment fonctionne cette fonction RAG ?**
1. **Le Retriever** : On lui passe la question (`query`), il recherche les documents pertinents et les convertit en une liste de textes. Chaque texte représente le contenu d'une page (`page_content`).
2. **La Concaténation** : Nous allons concaténer cette liste de textes à l'aide d'un séparateur. Ce texte final devient notre **contexte**.
3. **La Génération** : On injecte ce contexte et la question dans le `prompt_template`. On appelle ensuite la méthode `invoke()` et on retourne le contenu généré.

Voici donc la fonction qui nous intéresse. Pour pouvoir la tester, il vous suffira de faire appel à `RAG()` à chaque fois que vous voudrez lui poser une question.



In [52]:
def RAG(query,llm=llm,prompt_template=prompt_template):
    context_docs=retriever.invoke(query)
    context_list=[d.page_content for d in context_docs]
    context_for_query=". ".join(context_list)   
    prompt=prompt_template.format(context=context_for_query, question=query)
    resp=llm.invoke(prompt)
    return resp.content

### Appel de RAG 

In [53]:
resp=RAG(" qu'elle est son age ? ")
print(display(Markdown(resp)))

Elle est née le 7 novembre 2003, donc en décembre 2025 elle a 22 ans.

None


# 7. Évaluation de la réponse du LLM


## 7.1 Architecture

![Logo](./rag_juge.gif)

In [54]:
#question ou query 
user_input=" qu'elle est son age ? "

Récupérer le context 

In [55]:
relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)

In [56]:
user_message_template="""
###Question
{question}
###Context
{context}
###Answer
{answer}
"""

In [58]:
# Réponse d'un RAG
answer=RAG(" qu'elle est son age ? ")
print(display(Markdown(answer)))

Elle est née le 7 novembre 2003, donc en décembre 2025 elle a 22 ans.

None


## 7.2- *Groundness*

Le prompt pour le juge 

In [59]:
groundness_rater_system_message="""
Vous êtes chargé d'évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d'IA pour générer la réponse, ainsi qu'une réponse générée par l'IA à la question.
Dans l'entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par L'IA commencera par ###Answer.
Critères d'évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1- La métrique n'est pas respectée du tout
2- La métrique n'est respectée que dans une mesure limitée
3- La métrique est respectée dans une bonne mesure
4- La métrique est respectée en grande partie
5- La métrique est entièrement respectée

Métrique :

La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions:

Écrivez d'abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d'évaluation et attribuer un score.
"""

In [60]:
# Le juge doit être plus puissant que le LLM du RAG
from langchain_groq import ChatGroq
groundness_checker = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

In [61]:
def evaluate(system_message, user_message_template,question, model=groundness_checker):
    retrieved_chunks=retriever.invoke(question)
    context_list=[d.page_content for d in retrieved_chunks]
    context=". ".join(context_list)
    answer=RAG(question) 
    prompt=f"""
     { system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
     """
    juge_response=model.invoke(prompt)
    return juge_response.content



In [62]:
#évaluer
resp=evaluate(groundness_rater_system_message, user_message_template, user_input)
print(display(Markdown(resp)))

**Étapes pour évaluer la réponse selon la métrique « la réponse doit être dérivée uniquement des informations présentées dans le contexte »**

1. **Identifier les informations pertinentes du contexte**  
   - Rechercher dans le texte tout élément qui pourrait répondre à la question.  
   - Ici : `date‑naissance :07/11/2003`.

2. **Déterminer ce qui est demandé dans la question**  
   - La question « qu'elle est son âge ? » requiert l’âge actuel (ou à une date précise) de la personne.

3. **Comparer la réponse générée avec les informations du contexte**  
   - Vérifier que chaque donnée ou calcul présenté dans la réponse provient directement du contexte.  
   - Si la réponse nécessite un calcul (ex. : âge), le point de référence (date à laquelle l’âge est calculé) doit être explicitement présent dans le contexte ou être implicitement déductible sans ajouter d’information externe.

4. **Détecter toute information supplémentaire ou toute supposition**  
   - Rechercher des éléments qui ne figurent pas dans le contexte (ex. : date actuelle, hypothèses sur le futur, connaissances générales non mentionnées).

5. **Évaluer le degré de conformité**  
   - **Score 5** : toutes les informations et le raisonnement proviennent du contexte, aucune donnée externe n’est ajoutée.  
   - **Score 4** : la réponse utilise presque exclusivement le contexte, mais inclut un détail mineur non présent.  
   - **Score 3** : la majorité du contenu vient du contexte, mais une partie importante (ex. : un calcul basé sur une date non fournie) repose sur une supposition.  
   - **Score 2** : la réponse ne respecte le contexte que de façon limitée ; elle mélange correctement une donnée du contexte avec des informations externes essentielles.  
   - **Score 1** : la réponse ne se base pas du tout sur le contexte.

---

**Application de ces étapes à l’exemple fourni**

1. **Informations du contexte**  
   - La seule donnée pertinente est la date de naissance : `07/11/2003`.

2. **Ce qui est demandé**  
   - L’âge de la personne (au moment où la question est posée).

3. **Analyse de la réponse**  
   - La réponse indique : « Elle est née le 7 novembre 2003, donc en décembre 2025 elle a 22 ans. »  
   - La partie « Elle est née le 7 novembre 2003 » provient bien du contexte.  
   - Le calcul « en décembre 2025 elle a 22 ans » nécessite de connaître la date de référence (décembre 2025). Cette date n’apparaît nulle part dans le contexte fourni.

4. **Informations supplémentaires / suppositions**  
   - La référence à « décembre 2025 » est une supposition externe.  
   - Le calcul de l’âge (22 ans) dépend de cette date hypothétique et n’est donc pas dérivé uniquement du contexte.

5. **Évaluation du respect de la métrique**  
   - La réponse utilise correctement la date de naissance du contexte, mais l’âge donné repose sur une date de référence qui n’est pas mentionnée dans le contexte.  
   - Ainsi, la conformité est limitée : seule une partie de la réponse (la date de naissance) est tirée du contexte, tandis que le résultat chiffré (22 ans) dépend d’une information externe.

**Score attribué : 2** (la métrique est respectée seulement dans une mesure limitée).

None


### Échelle de notation pour la fidélité au contexte (Faithfulness)


| Score | Signification | Description |
| :--- | :--- | :--- |
| **5 / 5** | **Excellent (Parfait)** | La réponse utilise **uniquement** les informations du contexte. Aucune hallucination, aucune extrapolation. |
| **4 / 5** | **Bon** | La réponse est correcte et s'appuie sur le contexte, mais la formulation pourrait être légèrement plus précise. |
| **3 / 5** | **Moyen** | La réponse est partiellement vraie, mais elle omet un détail important du contexte ou manque de clarté. |
| **2 / 5** | **Médiocre** | La réponse contient des extrapolations ou des informations non vérifiables avec le contexte fourni. |
| **1 / 5** | **Mauvais (Hallucination)** | La réponse est totalement hors-sujet, fausse, ou invente des données qui ne sont pas du tout dans le contexte. |
